In [1]:
import pandas as pd 
import numpy as np
from pathlib import Path
import matplotlib.pyplot as plt
import seaborn as sns
import os
import re

In [51]:
relevant_cols = ["Gene", "Sample ID", "Cancer Type Detailed", "Mutation Type", "Variant Type", "HGVSc", "MS", 
        "Protein Change", "Functional Impact"]

data_path = os.path.abspath("../data/tcga_data2.csv")
raw_df = pd.read_csv(data_path, sep = '\t', low_memory = False)
raw_df.head(1)

,Gene,Study of Origin,Sample ID,Cancer Type,Cancer Type Detailed,Protein Change,Annotation,Custom Driver,Custom Driver Tiers,Functional Impact,...,Tumor Type,Used in Genomic Analysis,Vascular invasion indicator,Vessel Invasion,Vial number,Patient's Vital Status,Patient Weight,WGD,Winter Hypoxia Score,Year of Diagnosis
0,APC,"Colorectal Adenocarcinoma (TCGA, Firehose Legacy)",TCGA-AA-A010-01,Colorectal Cancer,Colon Adenocarcinoma,A2D,"OncoKB: Unknown, level NA, resistance NA;reVUE...",NaN,NaN,MutationAssessor: NA;SIFT: impact: deleterious...,...,NaN,NaN,NO,NaN,A,NaN,NaN,NaN,NaN,NaN


In [60]:
def groupby_age(df):
    cols = np.array(df.columns.tolist())
    pattern = re.compile(r'\bage\b', flags = re.IGNORECASE)
    age_cols = np.array([col for col in cols if pattern.search(col) and "Diagnosis" in col])
    df.loc[:, 'Age'] = df[age_cols].bfill(axis = 1).iloc[:, 0]
    df = df.drop(columns = age_cols)
    return df

In [35]:
def columns(df, relevant_cols):
    necessary_cols = ['Sample ID','HGVSc']
    if set(relevant_cols).intersection(necessary_cols) == set(necessary_cols):
        good_cols = np.concatenate((relevant_cols, ['Age']))
        new_df = df[good_cols].copy()
        return new_df
    else:
        raise ValueError("Relevant columns must contain sample ID and HGVSc")

In [12]:
def remove_duplicates(df):
    df = df.dropna(subset = ["Age"], axis = 0)
    df = df.drop_duplicates(subset = ["Sample ID", "HGVSc"], keep = "first")
    return df

In [42]:
pipeline = (raw_df
    .pipe(groupby_age)
    .pipe(columns, relevant_cols = relevant_cols)
    .pipe(remove_duplicates)
           )

pipeline.head(5)

,Gene,Sample ID,Cancer Type Detailed,Mutation Type,Variant Type,HGVSc,MS,Protein Change,Functional Impact,Age
0,APC,TCGA-AA-A010-01,Colon Adenocarcinoma,Missense_Mutation,SNP,ENST00000257430.4:c.5C>A,Somatic,A2D,MutationAssessor: NA;SIFT: impact: deleterious...,46.0
3,APC,TCGA-EI-6917-01,Rectal Adenocarcinoma,Missense_Mutation,SNP,ENST00000257430.4:c.7G>A,.,A3T,MutationAssessor: NA;SIFT: impact: deleterious...,33.0
4,APC,coadread_dfci_2016_3235,Colorectal Adenocarcinoma,Missense_Mutation,SNP,ENST00000257430.4:c.67C>A,NaN,L23I,MutationAssessor: NA;SIFT: impact: deleterious...,86.0
5,APC,coadread_dfci_2016_2367,Colorectal Adenocarcinoma,Nonsense_Mutation,SNP,ENST00000257430.4:c.70C>T,NaN,R24*,MutationAssessor: NA;SIFT: NA;Polyphen-2: NA;A...,66.0
6,APC,P-0006759-T01-IM5,Colorectal Adenocarcinoma,Missense_Mutation,SNP,ENST00000257430.4:c.100C>T,NaN,L34F,MutationAssessor: NA;SIFT: impact: deleterious...,51.0


In [59]:
def strip_strings(df):
    df.columns = df.columns.str.strip()
    return df

In [6]:
def early_onset_df(dataframe, early_age):
    df = dataframe["Early Onset"] = dataframe["Age"] < early_age
    return df

Trying to find a null cohort

In [62]:
data_path = os.path.abspath("../data/lung_data.csv")
lung_df = pd.read_csv(data_path, low_memory = False)
new_pipeline = (lung_df
    .pipe(strip_strings)
    .pipe(groupby_age)
    .pipe(columns, relevant_cols = relevant_cols)
    .pipe(remove_duplicates)
           )
new_pipeline.head(5)

,Gene,Sample ID,Cancer Type Detailed,Mutation Type,Variant Type,HGVSc,MS,Protein Change,Functional Impact,Age
1,SFRP1,TCGA-44-6778-01,Lung Adenocarcinoma,Nonsense_Mutation,SNP,NaN,Somatic,W287*,MutationAssessor: NA;SIFT: NA;Polyphen-2: NA;A...,59.0
2,SFRP1,TCGA-L9-A7SV-01,Lung Adenocarcinoma,Frame_Shift_Del,DEL,NaN,Somatic,I126Sfs*44,MutationAssessor: NA;SIFT: NA;Polyphen-2: NA;A...,69.0
3,SFRP1,TCGA-44-6778-01,Lung Adenocarcinoma,Nonsense_Mutation,SNP,ENST00000220772.3:c.860G>A,.,W287*,MutationAssessor: NA;SIFT: NA;Polyphen-2: NA;A...,59.0
5,SFRP1,TCGA-L9-A7SV-01,Lung Adenocarcinoma,Frame_Shift_Del,DEL,ENST00000220772.3:c.375del,Somatic,I126Sfs*44,MutationAssessor: NA;SIFT: NA;Polyphen-2: NA;A...,69.0
6,SFRP1,LUAD-CHTN-Z4716A,Lung Adenocarcinoma,Missense_Mutation,SNP,ENST00000220772.3:c.728T>A,Unknown,L243Q,"MutationAssessor: impact: medium, score: 5.514...",64.0
